In [ ]:
"""
نظام NuSec-CipherWeave
برنامج تشفير وفك تشفير باستخدام مصفوفة 256×256
للتشغيل على Google Colab

"""

import numpy as np
import pandas as pd
from tabulate import tabulate

# ============================================
# 1. بناء المصفوفة 256×256
# ============================================

def build_encryption_matrix():
    """
    بناء المصفوفة 256×256 حسب الفكرة:
    - الصف الأول: 0,1,2,...,255
    - الصف الثاني: 1,2,3,...,254,0
    - وهكذا حتى الصف 256: 255,0,1,...,254
    """
    matrix = np.zeros((256, 256), dtype=int)

    for i in range(256):
        for j in range(256):
            # الصف i يبدأ من i ثم يزيد بشكل دوري
            matrix[i][j] = (j + i) % 256

    return matrix

def build_decryption_matrix(enc_matrix):
    """
    بناء مصفوفة لفك التشفير (للبحث السريع عن المواقع)
    """
    dec_matrix = {}
    for row in range(256):
        for col in range(256):
            value = enc_matrix[row][col]
            dec_matrix[(row, value)] = col
    return dec_matrix

# ============================================
# 2. دوال التشفير وفك التشفير
# ============================================

def encrypt_message(plain_text, key, matrix):
    """
    تشفير النص السري باستخدام المفتاح

    المعطيات:
    - plain_text: النص المراد تشفيره (string)
    - key: مفتاح التشفير (string)
    - matrix: مصفوفة التشفير 256×256

    المخرجات:
    - cipher_text: النص المشفر (string)
    """
    # تحويل النص والمفتاح إلى قيم ASCII
    plain_bytes = [ord(c) for c in plain_text]
    key_bytes = [ord(c) for c in key]

    cipher_bytes = []
    key_len = len(key_bytes)

    print("\n" + "="*60)
    print("📝 عملية التشفير")
    print("="*60)

    for i, p in enumerate(plain_bytes):
        # الحصول على حرف المفتاح الحالي (مع إعادة الاستخدام)
        k = key_bytes[i % key_len]

        # البحث عن موقع k في الصف الأول من المصفوفة
        # الصف الأول هو row = 0
        col_of_k = None
        for col in range(256):
            if matrix[0][col] == k:
                col_of_k = col
                break

        # البحث عن موقع p في العمود الأول من المصفوفة
        # العمود الأول هو col = 0
        row_of_p = None
        for row in range(256):
            if matrix[row][0] == p:
                row_of_p = row
                break

        # الحصول على قيمة التشفير من التقاطع
        cipher_value = matrix[row_of_p][col_of_k]
        cipher_bytes.append(cipher_value)

        # طباعة التفاصيل (للمتابعة)
        print(f"\n📌 الحرف {i+1}:")
        print(f"   النص الأصلي: '{chr(p)}' (ASCII: {p}) → الصف: {row_of_p}")
        print(f"   مفتاح: '{chr(k)}' (ASCII: {k}) → العمود: {col_of_k}")
        print(f"   ✅ النص المشفر: '{chr(cipher_value)}' (ASCII: {cipher_value})")

    # تحويل القيم المشفرة إلى نص
    cipher_text = ''.join([chr(c) for c in cipher_bytes])

    return cipher_text, cipher_bytes

def decrypt_message(cipher_text, key, matrix, dec_lookup):
    """
    فك تشفير النص المشفر

    المعطيات:
    - cipher_text: النص المشفر (string)
    - key: مفتاح التشفير (string)
    - matrix: مصفوفة التشفير 256×256
    - dec_lookup: قاموس للبحث السريع (row, value) -> col

    المخرجات:
    - plain_text: النص الأصلي (string)
    """
    cipher_bytes = [ord(c) for c in cipher_text]
    key_bytes = [ord(c) for c in key]

    plain_bytes = []
    key_len = len(key_bytes)

    print("\n" + "="*60)
    print("🔓 عملية فك التشفير")
    print("="*60)

    for i, c in enumerate(cipher_bytes):
        # الحصول على حرف المفتاح الحالي
        k = key_bytes[i % key_len]

        # البحث عن موقع k في الصف الأول
        col_of_k = None
        for col in range(256):
            if matrix[0][col] == k:
                col_of_k = col
                break

        # البحث في الصف col_of_k عن القيمة c
        row_of_p = None
        for row in range(256):
            if matrix[row][col_of_k] == c:
                row_of_p = row
                break

        # الحصول على النص الأصلي من العمود الأول في هذا الصف
        original_value = matrix[row_of_p][0]
        plain_bytes.append(original_value)

        # طباعة التفاصيل
        print(f"\n📌 الحرف {i+1}:")
        print(f"   النص المشفر: '{chr(c)}' (ASCII: {c})")
        print(f"   مفتاح: '{chr(k)}' (ASCII: {k}) → العمود: {col_of_k}")
        print(f"   ✅ النص الأصلي: '{chr(original_value)}' (ASCII: {original_value})")

    plain_text = ''.join([chr(b) for b in plain_bytes])
    return plain_text

# ============================================
# 3. دوال إضافية للعرض والتحليل
# ============================================

def display_matrix_sample(matrix, rows=10, cols=10):
    """
    عرض عينة من المصفوفة
    """
    print("\n" + "="*60)
    print("📊 عينة من مصفوفة التشفير (أول 10×10 خلايا)")
    print("="*60)

    sample = matrix[:rows, :cols]
    df = pd.DataFrame(sample)
    print(tabulate(df, headers=[f"Col {i}" for i in range(cols)],
                   showindex=[f"Row {i}" for i in range(rows)],
                   tablefmt="grid"))

def analyze_matrix(matrix):
    """
    تحليل إحصائي للمصفوفة
    """
    print("\n" + "="*60)
    print("📈 تحليل إحصائي للمصفوفة")
    print("="*60)

    # جميع القيم من 0-255 تظهر مرة واحدة في كل صف وكل عمود
    unique_in_row = all(len(set(matrix[row])) == 256 for row in range(256))
    unique_in_col = all(len(set(matrix[:, col])) == 256 for col in range(256))

    print(f"✓ كل صف يحتوي على جميع القيم 0-255: {unique_in_row}")
    print(f"✓ كل عمود يحتوي على جميع القيم 0-255: {unique_in_col}")
    print(f"✓ حجم المصفوفة: {matrix.shape}")

def save_results(plain_text, cipher_text, decrypted_text, key):
    """
    حفظ النتائج في ملف
    """
    with open('encryption_results.txt', 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("نتائج التشفير\n")
        f.write("="*60 + "\n\n")
        f.write(f"المفتاح المستخدم: {key}\n\n")
        f.write(f"النص الأصلي: {plain_text}\n")
        f.write(f"النص المشفر: {cipher_text}\n")
        f.write(f"النص بعد فك التشفير: {decrypted_text}\n\n")

        # عرض القيم الرقمية
        f.write("القيم الرقمية:\n")
        f.write(f"ASCII الأصلي: {[ord(c) for c in plain_text]}\n")
        f.write(f"ASCII المشفر: {[ord(c) for c in cipher_text]}\n")
        f.write(f"ASCII بعد فك التشفير: {[ord(c) for c in decrypted_text]}\n")

    print("\n" + "="*60)
    print("💾 تم حفظ النتائج في ملف 'encryption_results.txt'")
    print("="*60)

# ============================================
# 4. الوظيفة الرئيسية
# ============================================

def main():
    print("\n" + "="*60)
    print("🔐 برنامج التشفير باستخدام مصفوفة 256×256")
    print("="*60)

    # بناء المصفوفات
    print("\n🔄 جاري بناء مصفوفة التشفير...")
    enc_matrix = build_encryption_matrix()
    dec_lookup = build_decryption_matrix(enc_matrix)
    print("✅ تم بناء المصفوفة بنجاح!")

    # عرض تحليل المصفوفة
    analyze_matrix(enc_matrix)
    display_matrix_sample(enc_matrix)

    # إدخال البيانات من المستخدم
    print("\n" + "="*60)
    print("📝 إدخال البيانات")
    print("="*60)

    # النص السري
    plain_text = input("\n🔤 أدخل النص المراد تشفيره: ").strip()
    while len(plain_text) == 0:
        plain_text = input("⚠️ النص لا يمكن أن يكون فارغاً. أعد المحاولة: ").strip()

    # مفتاح التشفير
    key = input("🔑 أدخل مفتاح التشفير: ").strip()
    while len(key) == 0:
        key = input("⚠️ المفتاح لا يمكن أن يكون فارغاً. أعد المحاولة: ").strip()

    # عرض معلومات الإدخال
    print(f"\n📌 النص الأصلي: '{plain_text}' (طول {len(plain_text)} حرف)")
    print(f"📌 المفتاح: '{key}' (طول {len(key)} حرف)")

    # تأكيد العملية
    confirm = input("\n❓ هل تريد بدء عملية التشفير؟ (y/n): ").lower()
    if confirm != 'y':
        print("❌ تم إلغاء العملية.")
        return

    # تنفيذ التشفير
    try:
        cipher_text, cipher_bytes = encrypt_message(plain_text, key, enc_matrix)

        print("\n" + "="*60)
        print("✨ نتائج التشفير")
        print("="*60)
        print(f"\n🔐 النص المشفر: '{cipher_text}'")
        print(f"📊 القيم المشفرة (ASCII): {cipher_bytes}")

        # تنفيذ فك التشفير للتحقق
        decrypted_text = decrypt_message(cipher_text, key, enc_matrix, dec_lookup)

        print("\n" + "="*60)
        print("✅ نتائج التحقق")
        print("="*60)
        print(f"\n🔓 النص بعد فك التشفير: '{decrypted_text}'")

        if plain_text == decrypted_text:
            print("\n🎉 نجاح! فك التشفير صحيح 100%")
        else:
            print("\n⚠️ تحذير: فك التشفير لا يتطابق مع النص الأصلي!")

        # حفظ النتائج
        save_results(plain_text, cipher_text, decrypted_text, key)

    except Exception as e:
        print(f"\n❌ حدث خطأ: {e}")
        print("يرجى التأكد من صحة المدخلات")

# ============================================
# 5. تشغيل البرنامج
# ============================================

if __name__ == "__main__":
    main()

    # خيار لعرض المصفوفة كاملة (إذا احتجت)
    print("\n" + "="*60)
    print("💡 ملاحظات:")
    print("="*60)
    print("""
1. يمكنك استعراض المصفوفة الكاملة عن طريق إضافة الكود التالي:
   >>> matrix = build_encryption_matrix()
   >>> print(matrix)

2. المصفوفة هي 256×256 وكل صف يحتوي على القيم 0-255 مرتبة بشكل دوري.

3. التشفير يعتمد على موقع الحرف في الصف الأول والعمود الأول.

4. يتم إعادة استخدام المفتاح بشكل دوري إذا كان أقصر من النص.

5. جميع الأحرف (بما فيها المسافات والرموز) يتم تشفيرها بشكل صحيح.
    """)
